# EmployeeEdge — Preprocessing & Feature Engineering

## What this step does
Raw data cannot go into a machine learning model directly. This notebook converts it into a **model-ready format**:
- Drops columns that carry no information
- Encodes text columns into numbers
- Scales numerical columns
- Splits data into **train** (to learn patterns) and **test** (to test on unseen data)

## The golden rule — NO DATA LEAKAGE
Anything we "learn" from the data (e.g., the average salary, the list of job roles) must be learned **ONLY from the training set**. If we learn it from the whole dataset, the test set stops being "unseen" and our results become a lie.

We solve this with a **sklearn Pipeline**, which we also save for reuse in the app later.

---
## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from collections import Counter

RANDOM_STATE = 42
print('Libraries loaded successfully.')

Libraries loaded successfully.


---
## 2. Load the Data

We re-read the **raw** CSV every time. We never edit the raw file — all changes happen in a copy, so we can always trace back to the original data.

In [2]:
df = pd.read_csv('../data/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(f'Loaded: {df.shape[0]} rows x {df.shape[1]} columns')
df.head(3)

Loaded: 1470 rows x 35 columns


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0


---
## 3. Drop Useless Columns

From our EDA we found 4 columns that add **zero predictive value**:

| Column | Why drop it |
|---|---|
| `EmployeeCount` | Same value (1) for every employee — no variation |
| `Over18` | Same value (Y) for everyone |
| `StandardHours` | Same value (80) for everyone |
| `EmployeeNumber` | Just an ID — not a feature an HR team could act on |

**Why:** A model can't learn from a column where every row is identical. These only add noise.

In [3]:
drop_cols = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df_clean = df.drop(columns=drop_cols)

print(f'Dropped {len(drop_cols)} columns: {drop_cols}')
print(f'Remaining: {df_clean.shape[1]} columns (features + target)')

numerical_cols = df_clean.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df_clean.select_dtypes(include='object').columns.tolist()
categorical_cols.remove('Attrition')

print(f'\nNumerical features ({len(numerical_cols)}):')
print(numerical_cols)
print(f'\nCategorical features ({len(categorical_cols)}):')
print(categorical_cols)

Dropped 4 columns: ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
Remaining: 31 columns (features + target)

Numerical features (23):
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

Categorical features (7):
['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']


---
## 4. Separate Features (X) and Target (y)

- **X** = the features the model reads to make a prediction
- **y** = the answer we're predicting (Attrition: 1 = left, 0 = stayed)

**Why:** Models learn the mapping `X -> y`. Keeping them separate makes the workflow clean and prevents accidentally using y as a feature.

In [4]:
X = df_clean.drop(columns=['Attrition'])
y = df_clean['Attrition'].map({'Yes': 1, 'No': 0})

print(f'Features (X): {X.shape[0]} rows x {X.shape[1]} columns')
print(f'Target (y):   Attrition -> {dict(Counter(y))}')

Features (X): 1470 rows x 30 columns
Target (y):   Attrition -> {1: 237, 0: 1233}


---
## 5. Train / Test Split (Stratified)

We hold out **20% of the data** as the test set — data the model has never seen. Everything is learned on the remaining 80% (train).

### Why `stratify=y`?
The dataset is imbalanced (16% left). A plain random split could give: train 18% left, test 12% left — a skewed test set that misrepresents reality.

**Stratification** keeps the same ratio (16% left) in *both* splits.

### Why `random_state=42`?
Splitting is random. A fixed seed makes the split **reproducible** — run it twice, get the same answer. (42 is just the traditional choice.)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples')
print(f'\nAttrition rate in each split (should all be ~16.1%):')
print(f'  Full dataset: {(y == 1).mean() * 100:.2f}%')
print(f'  Train        : {(y_train == 1).mean() * 100:.2f}%')
print(f'  Test         : {(y_test == 1).mean() * 100:.2f}%')
print(f'\nTrain class distribution: {dict(Counter(y_train))}')

Train: 1176 samples  |  Test: 294 samples

Attrition rate in each split (should all be ~16.1%):
  Full dataset: 16.12%
  Train        : 16.16%
  Test         : 15.99%

Train class distribution: {0: 986, 1: 190}


---
## 6. Dealing with Class Imbalance

Only 237 of 1,470 employees left. If we train naively, the model can reach ~84% accuracy by predicting "stayed" for everyone — a useless tool.

### Two common strategies
| Strategy | How it works | Trade-off |
|---|---|---|
| **Class weights** | The model multiplies the error *for each left-case* by ~5x, forcing it to care about the minority class | No synthetic data, clean — our choice |
| **SMOTE** | Generates fake minority samples by interpolating between real ones | Extra complexity; can introduce noise |

**Decision:** We will use `class_weight='balanced'` inside the models in the next step. We'll also evaluate with **Precision / Recall / F1 / AUC** instead of plain accuracy.

---
## 7. Build the Preprocessing Pipeline

We use sklearn's `ColumnTransformer` — a single object that applies different transformations to different column types:

| Transformer | Applied to | Why |
|---|---|---|
| `StandardScaler` | 23 numerical cols | Centers values around 0 with std=1. Required for Logistic Regression (distance-based) so big numbers (income) don't dominate small numbers (age) |
| `OneHotEncoder` | 7 categorical cols | Turns text like `JobRole` into binary columns (is Sales Executive? is Lab Technician? ...). `handle_unknown='ignore'` makes the app robust to unseen values |

### Why this is the right approach
- We `fit` ONLY on train, then `transform` — zero data leakage
- We save this object (`preprocessor.joblib`) and reuse it **exactly** in the app — same transformations in dev and production = no bugs

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print(f'Transformed TRAIN shape: {X_train_processed.shape}')
print(f'Transformed TEST shape : {X_test_processed.shape}')
print(f'Original features: {X.shape[1]}  ->  Encoded features: {len(feature_names)}')
print('\n=== ENCODED FEATURE NAMES (sample) ===')
print(feature_names[:20])

Transformed TRAIN shape: (1176, 51)
Transformed TEST shape : (294, 51)
Original features: 30  ->  Encoded features: 51

=== ENCODED FEATURE NAMES (sample) ===
['num__Age' 'num__DailyRate' 'num__DistanceFromHome' 'num__Education'
 'num__EnvironmentSatisfaction' 'num__HourlyRate' 'num__JobInvolvement'
 'num__JobLevel' 'num__JobSatisfaction' 'num__MonthlyIncome'
 'num__MonthlyRate' 'num__NumCompaniesWorked' 'num__PercentSalaryHike'
 'num__PerformanceRating' 'num__RelationshipSatisfaction'
 'num__StockOptionLevel' 'num__TotalWorkingYears'
 'num__TrainingTimesLastYear' 'num__WorkLifeBalance' 'num__YearsAtCompany']


---
## 8. Inspect the Transformed Data

Quick sanity check:
- Numerical columns should have **mean ~ 0, std ~ 1** after scaling
- Categorical columns should only contain **0 and 1** (one-hot)

In [7]:
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_train_df.head()

,num__Age,num__DailyRate,num__DistanceFromHome,num__Education,num__EnvironmentSatisfaction,num__HourlyRate,num__JobInvolvement,num__JobLevel,num__JobSatisfaction,num__MonthlyIncome,...,cat__JobRole_Manufacturing Director,cat__JobRole_Research Director,cat__JobRole_Research Scientist,cat__JobRole_Sales Executive,cat__JobRole_Sales Representative,cat__MaritalStatus_Divorced,cat__MaritalStatus_Married,cat__MaritalStatus_Single,cat__OverTime_No,cat__OverTime_Yes
0,1.090194,1.049455,-0.899915,1.064209,-0.658710,-0.908436,1.795282,1.762189,-0.647997,2.026752,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,-1.634828,-0.523449,-0.899915,-1.855332,0.260202,1.694111,0.373564,-0.986265,1.153526,-0.864408,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,0.981193,-0.992080,-0.777610,-1.855332,-1.577622,-0.662913,0.373564,1.762189,0.252765,2.347706,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
3,-1.307825,-0.453653,0.445433,-1.855332,-0.658710,-1.252169,0.373564,-0.986265,0.252765,-0.956202,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4,0.654191,0.491086,-0.043784,2.037390,1.179114,0.319180,0.373564,-0.070114,0.252765,-0.185956,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [8]:
num_feats = [f for f in feature_names if f.startswith('num__')]
cat_feats = [f for f in feature_names if f.startswith('cat__')]

print('Numerical features after scaling (mean~0, std~1):')
print(X_train_df[num_feats].describe().T[['mean', 'std']].head(10).round(3))

print(f'\nCategorical features: {len(cat_feats)} one-hot columns')
print('Example - JobRole columns (only 0s and 1s):')
role_cols = [c for c in cat_feats if 'JobRole' in c]
print(X_train_df[role_cols].head())

Numerical features after scaling (mean~0, std~1):


                              mean  std
num__Age                      -0.0  1.0
num__DailyRate                 0.0  1.0
num__DistanceFromHome         -0.0  1.0
num__Education                 0.0  1.0
num__EnvironmentSatisfaction   0.0  1.0
num__HourlyRate               -0.0  1.0
num__JobInvolvement            0.0  1.0
num__JobLevel                  0.0  1.0
num__JobSatisfaction           0.0  1.0
num__MonthlyIncome            -0.0  1.0

Categorical features: 28 one-hot columns
Example - JobRole columns (only 0s and 1s):
   cat__JobRole_Healthcare Representative  cat__JobRole_Human Resources  \
0                                     0.0                           0.0   
1                                     0.0                           0.0   
2                                     0.0                           0.0   
3                                     0.0                           0.0   
4                                     0.0                           0.0   

   cat__JobRole_Laborat

---
## 9. Save the Preprocessor

We save the **fitted** pipeline with `joblib`. Later:
- The modeling notebook loads it to transform test data consistently
- The app loads it to transform user input the **exact same way** the model expects

In [9]:
os.makedirs('../outputs/models', exist_ok=True)
joblib.dump(preprocessor, '../outputs/models/preprocessor.joblib')
print('Saved preprocessor -> outputs/models/preprocessor.joblib')

Saved preprocessor -> outputs/models/preprocessor.joblib


---
## 10. Preprocessing Summary

### What we did
1. Dropped 4 useless columns (3 constant + 1 ID)
2. Separated features (X) from target (y)
3. **Stratified** 80/20 train/test split — preserves the 16% attrition ratio in both
4. Built a `ColumnTransformer`:
   - Numerical (23): `StandardScaler`
   - Categorical (7): `OneHotEncoder`
5. Fit on train only — **no data leakage**
6. Saved the fitted preprocessor for reuse

### What's next
**Step 4 — Modeling:**
- Train Logistic Regression, Gradient Boosting, and XGBoost (with `class_weight='balanced'`)
- Tune hyperparameters with cross-validation
- Compare using Precision / Recall / F1 / AUC-ROC
- Explain predictions with SHAP